In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import bilby 
import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")

In [ ]:
outdir = "outdir"
label = "two_param_estimation"
bilby.core.utils.setup_logger(outdir=outdir, label=label,)

In [ ]:
injection_parameters = dict(
    mass_1=15,
    mass_2=5,
    a_1=0,
    a_2=0, 
    tilt_1=0.0, 
    tilt_2=0.0, 
    phi_12=0.0,
    phi_jl=0.0, 
    theta_jn=0.4,
    luminosity_distance=40,
    ra=1.375,
    dec=-1.2108,
    psi=2.659, 
    phase=1.3,
    geocent_time=1126259642.413)

def get_chirp_mass_mass_ratio(parameters):
    converted_parameters = parameters.copy()
    converted_parameters["mass_ratio"] = bilby.gw.conversion.component_masses_to_mass_ratio(injection_parameters["mass_1"],injection_parameters["mass_2"])
    converted_parameters["chirp_mass"] = bilby.gw.conversion.component_masses_to_chirp_mass(injection_parameters["mass_1"],injection_parameters["mass_2"])
    return converted_parameters


injection_parameters = get_chirp_mass_mass_ratio(injection_parameters)

In [ ]:
sampling_frequency = 2048 
                          
minimum_frequency = 20 
maximum_frequency = 4400/(injection_parameters['mass_1']+injection_parameters['mass_2'])

duration = bilby.gw.detector.get_safe_signal_duration(
    injection_parameters['mass_1'],
    injection_parameters['mass_2'],
    injection_parameters['a_1'],
    injection_parameters['a_2'],
    injection_parameters['tilt_1'],
    injection_parameters['tilt_2'],
    flow=minimum_frequency)

end_time = injection_parameters["geocent_time"] + 2
start_time = end_time - duration 

In [ ]:
waveform_arguments = dict(
    waveform_approximant="TaylorF2",
    reference_frequency=50.0, 
    minimum_frequency=minimum_frequency,
    maximum_frequency=maximum_frequency,
)

waveform_generator = bilby.gw.WaveformGenerator(
    start_time = start_time,
    duration=duration,
    sampling_frequency=sampling_frequency,
    frequency_domain_source_model=bilby.gw.source.lal_binary_black_hole,
    parameter_conversion=bilby.gw.conversion.convert_to_lal_binary_black_hole_parameters,
    waveform_arguments=waveform_arguments,
    )

In [ ]:
ifos = bilby.gw.detector.InterferometerList(["H1"])
 
ifos.set_strain_data_from_zero_noise( 
    sampling_frequency=sampling_frequency,
    duration=duration,
    start_time=start_time,
);

ifos.inject_signal(
    waveform_generator=waveform_generator, parameters=injection_parameters
);

for ifo in ifos:
    ifo.minimum_frequency = waveform_arguments["minimum_frequency"]
    ifo.maximum_frequency = waveform_arguments["maximum_frequency"]
   

In [ ]:
likelihood = bilby.gw.GravitationalWaveTransient(
    interferometers=ifos, 
    waveform_generator=waveform_generator 
)

In [ ]:
priors = bilby.gw.prior.PriorDict()

non_sampled_parameters =[
    "a_1",
    "a_2",
    "tilt_1",
    "tilt_2",
    "phi_12",
    "phi_jl",
    "psi",
    "luminosity_distance",
    "theta_jn",
    "ra",
    "dec",
    "geocent_time",
    "phase",
]
 
for key in non_sampled_parameters:
    priors[key] = injection_parameters[key]

priors["mass_ratio"] = bilby.core.prior.Uniform(minimum=0.125, maximum=1, name='mass_ratio', latex_label='$q$', unit=None, boundary=None,)
priors["chirp_mass"] = bilby.core.prior.Uniform(minimum=5, maximum=50, name='chirp_mass', latex_label='$\\mathcal{M}$', unit=None, boundary=None)

for key in priors.keys():
    print(key+": {}".format(str(priors[key])))

In [ ]:
result = bilby.run_sampler(
    likelihood=likelihood,
    priors=priors,
    sampler="dynesty",# this sampler uses nested sampling   
    nlive=500,#these settings are not sufficient for conversion 
    naccept=5,#these settings are not sufficient for conversion 
    sample="acceptance-walk",
    injection_parameters=injection_parameters,
    outdir=outdir,
    label=label,
    clean=True, # Switch this to False if you want to be able to checkpoint and restart your run. 
    conversion_function=bilby.gw.conversion.generate_all_bbh_parameters,
    result_class=bilby.gw.result.CBCResult,
)
result.plot_corner(save = False)

In [ ]:
np.savetxt("samples_chirp_mass_mass_ratio_q3.csv",np.transpose(np.array([result.posterior["chirp_mass"],result.posterior["mass_ratio"]])), delimiter=',')